# Türkçe → İngilizce Mini Makine Çevirisi (Seq2Seq)
Bu Google Colab defterinde adım adım temel bir **Encoder-Decoder** (LSTM tabanlı) makine çeviri modeli oluşturacağız. Her adımın altında detaylı açıklamalar yer alacaktır.

## 1. Gerekli Kütüphaneler
- **NumPy (`np`)**: Sayısal hesaplamalar.
- **TensorFlow/Keras**: Derin öğrenme modelleri için.
- **Tokenizer, pad_sequences**: Metni sayılara dönüştürmek için.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 2. Küçük Veri Seti
Elimizdeki Türkçe–İngilizce eşleştirilmiş cümleler:

In [2]:
pairs = [
    ("merhaba", "hello"),
    ("nasılsın", "how are you"),
    ("ben iyiyim", "i am fine"),
    ("seni seviyorum", "i love you"),
    ("teşekkür ederim", "thank you"),
    ("görüşürüz", "see you"),
]
# pairs listesi: (kaynak, hedef) formatında.

## 3. Giriş ve Çıkış Metinlerinin İşaretlenmesi
- `\t`: çeviri başlangıcı
- `\n`: çeviri sonu

In [3]:
input_texts = []
target_texts = []
for src, tgt in pairs:
    input_texts.append(src)
    target_texts.append('\t' + tgt + '\n')  # Başlangıç ve bitiş tokenleri eklenir
# input_texts: ['merhaba', ...]
# target_texts: ['\thello\n', ...]

## 4. Karakter Düzeyinde Tokenization
- Metni karakter bazlı sayılara çeviriyoruz.

In [4]:
# Kaynak dil (Türkçe)
input_tokenizer = Tokenizer(char_level=True)
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)
# Hedef dil (İngilizce)
target_tokenizer = Tokenizer(char_level=True)
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
# Her bir karaktere bir sayı atanır.

## 5. Padding: Dizileri Aynı Uzunluğa Getirme
Sinir ağları sabit boyut bekler; eksik yerlere 0 eklenir.

In [5]:
input_maxlen = max(len(seq) for seq in input_sequences)
target_maxlen = max(len(seq) for seq in target_sequences)
encoder_input_data = pad_sequences(input_sequences, maxlen=input_maxlen, padding='post')
decoder_input_data = pad_sequences([seq[:-1] for seq in target_sequences],
                                   maxlen=target_maxlen-1, padding='post')
decoder_target_data = pad_sequences([seq[1:] for seq in target_sequences],
                                    maxlen=target_maxlen-1, padding='post')

## 6. One-Hot Kodlama
Her karakter vektöre dönüştürülür (örneğin [0,0,1,0,...]).

In [6]:
num_encoder_tokens = len(input_tokenizer.word_index) + 1
num_decoder_tokens = len(target_tokenizer.word_index) + 1
decoder_target_onehot = tf.keras.utils.to_categorical(
    decoder_target_data, num_classes=num_decoder_tokens)

## 7. Modeli Kurma: Encoder-Decoder Mimarisi

In [7]:
# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, 64)(encoder_inputs)
encoder_lstm, state_h, state_c = LSTM(64, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(num_decoder_tokens, 64)(decoder_inputs)
decoder_lstm = LSTM(64, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## 8. Modelin Eğitilmesi
- `epochs=500`: Tüm veri üzerinde 500 kez geçilir.

In [8]:
model.fit([encoder_input_data, decoder_input_data],
          decoder_target_onehot,
          batch_size=2,
          epochs=500,
          verbose=1)

Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.0642 - loss: 3.0442
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2274 - loss: 3.0239
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3681 - loss: 2.9952
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3802 - loss: 2.9595
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3490 - loss: 2.9272
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3681 - loss: 2.8487
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3090 - loss: 2.7817
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3194 - loss: 2.5954
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2726 - loss: 2.5016
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3403 - loss: 2.2356
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2344 - loss: 2.4309
Epoch 12/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3594 - lo

## 9. Tahmin (Inference) için Encoder ve Decoder Modelleri

In [9]:
# Encoder model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder model
decoder_state_input_h = Input(shape=(64,))
decoder_state_input_c = Input(shape=(64,))
dec_emb2 = Embedding(num_decoder_tokens, 64)(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = Model(
    [decoder_inputs, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2, state_h2, state_c2]
)

## 10. Sayı → Karakter Haritaları

In [10]:
reverse_input_char_index = {v: k for k, v in input_tokenizer.word_index.items()}
reverse_target_char_index = {v: k for k, v in target_tokenizer.word_index.items()}

## 11. Çeviri Fonksiyonu

In [11]:
def translate_sequence(input_seq):
    # 1) Girdi cümleyi diziye çevir
    seq = input_tokenizer.texts_to_sequences([input_seq])
    seq = pad_sequences(seq, maxlen=input_maxlen, padding='post')
    # 2) Encoder'dan durumları al
    states_value = encoder_model.predict(seq)

    # 3) Decoder başlangıcı: '\t'
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['\t']

    decoded_sentence = ''
    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        if sampled_char == '\n' or len(decoded_sentence) > 20:
            break
        decoded_sentence += sampled_char
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence

## 12. Modeli Deneme

In [12]:
print("merhaba →", translate_sequence("merhaba"))
print("nasılsın →", translate_sequence("nasılsın"))
print("teşekkür ederim →", translate_sequence("teşekkür ederim"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
merhaba → hellll
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
nasılsın → how   e  ou
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━